# NeuroRAG )
### Uses pure NumPy biomarker vectors for embeddings

In [1]:
# CELL 1: IMPORTS — only standard packages
import mne
import numpy as np
import pandas as pd
from scipy.signal import welch
from pathlib import Path
import chromadb
from tqdm import tqdm
import warnings

mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')
print('All imports OK')

All imports OK


In [2]:
# CELL 2: CONFIG
DATASET_ROOT = Path(
    r"C:\Users\KARTHIKEYA\Desktop\An EEG Recordings Dataset for Mental Stress Detection"
)

CONDITIONS = {
    'CMPS'  : 'Complex Mathematical Problem solving (CMPS)',
    'HORROR': 'Horrer Video Stimulation',
    'MUSIC' : 'Participants Listening to Relaxing Music',
    'SCWT'  : 'Stroop Colour Word Test(SCWT)',
    'TMCT'  : 'Trier Mental Challenge Test (TMCT)'
}

EEG_CHANNELS = ['AF3', 'T7', 'Pz', 'T8', 'AF4']
SFREQ = 128

BANDS = {
    'delta': (0.5, 4),
    'theta': (4,   8),
    'alpha': (8,  13),
    'beta' : (13, 30),
    'gamma': (30, 45)
}

CHROMA_PATH = r"C:\Users\KARTHIKEYA\Desktop\neurorag_db"

# Verify folders
for key, folder in CONDITIONS.items():
    p = DATASET_ROOT / folder
    status = 'OK' if p.exists() else 'NOT FOUND'
    print(f'[{status}] {key}: {folder}')

[OK] CMPS: Complex Mathematical Problem solving (CMPS)
[OK] HORROR: Horrer Video Stimulation
[OK] MUSIC: Participants Listening to Relaxing Music
[OK] SCWT: Stroop Colour Word Test(SCWT)
[OK] TMCT: Trier Mental Challenge Test (TMCT)


In [3]:
# CELL 3: LOAD + FILTER EEG

def load_eeg(filepath):
    raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)
    available = [ch for ch in EEG_CHANNELS if ch in raw.ch_names]
    if not available:
        raise ValueError(f'No EEG channels in {filepath}')
    raw_eeg = raw.copy().pick(available)
    raw_eeg.filter(l_freq=0.5, h_freq=45.0, verbose=False)
    raw_eeg.notch_filter(freqs=50.0, verbose=False)
    return raw_eeg.get_data(), available

print('load_eeg defined')

load_eeg defined


In [4]:
# CELL 4: BAND POWERS

def compute_band_powers(data, ch_names):
    bp = {band: {} for band in BANDS}
    for ch_idx, ch_name in enumerate(ch_names):
        freqs, psd = welch(data[ch_idx], fs=SFREQ, nperseg=256, noverlap=128)
        for band, (lo, hi) in BANDS.items():
            mask = (freqs >= lo) & (freqs <= hi)
            bp[band][ch_name] = float(np.trapezoid(psd[mask], freqs[mask]))
    return bp

print('compute_band_powers defined')

compute_band_powers defined


In [5]:
# CELL 5: BIOMARKERS

def compute_biomarkers(bp, ch_names):
    bm  = {}
    eps = 1e-10

    theta_avg = float(np.mean(list(bp['theta'].values())))
    alpha_avg = float(np.mean(list(bp['alpha'].values())))
    beta_avg  = float(np.mean(list(bp['beta'].values())))

    bm['theta_alpha_ratio'] = theta_avg / (alpha_avg + eps)

    if 'AF3' in ch_names and 'AF4' in ch_names:
        bm['frontal_alpha_asymmetry'] = float(
            np.log(bp['alpha']['AF4'] + eps) - np.log(bp['alpha']['AF3'] + eps)
        )
    else:
        bm['frontal_alpha_asymmetry'] = 0.0

    bm['engagement_index'] = beta_avg / (alpha_avg + theta_avg + eps)

    faa = abs(bm['frontal_alpha_asymmetry'])
    tar = bm['theta_alpha_ratio']
    ei  = bm['engagement_index']
    bm['stress_score'] = round(
        0.4 * min(faa / 0.5, 1.0) +
        0.3 * min(tar / 2.0, 1.0) +
        0.3 * min(ei  / 1.5, 1.0), 4
    )
    return bm

print('compute_biomarkers defined')

compute_biomarkers defined


In [6]:
# CELL 6: BIOMARKER VECTOR EMBEDDING
# No sentence-transformers. No onnxruntime.
# We build a 35-dimensional vector directly from EEG features.
# Each dimension has a clear neuroscientific meaning.

def make_embedding(bp, bm, ch_names):
    """
    Build a 35-dim float vector from EEG biomarkers.
    Layout:
      [0-4]   : mean band powers (delta, theta, alpha, beta, gamma)
      [5-9]   : relative alpha per channel (AF3, T7, Pz, T8, AF4)
      [10-14] : relative beta per channel
      [15-19] : relative theta per channel
      [20-24] : beta/alpha ratio per channel
      [25]    : theta/alpha ratio (global)
      [26]    : frontal alpha asymmetry
      [27]    : engagement index
      [28]    : stress score
      [29-33] : absolute alpha power per channel
      [34]    : frontal beta asymmetry (AF4-AF3 log beta)
    """
    vec = []

    # [0-4] mean band powers
    for band in ['delta', 'theta', 'alpha', 'beta', 'gamma']:
        vec.append(float(np.mean(list(bp[band].values()))))

    # [5-24] per-channel relative powers (alpha, beta, theta)
    for band in ['alpha', 'beta', 'theta']:
        for ch in ['AF3', 'T7', 'Pz', 'T8', 'AF4']:
            if ch in ch_names:
                total = sum(bp[b][ch] for b in BANDS) + 1e-10
                vec.append(bp[band][ch] / total)
            else:
                vec.append(0.0)

    # [25-28] scalar biomarkers
    vec.append(bm['theta_alpha_ratio'])
    vec.append(bm['frontal_alpha_asymmetry'])
    vec.append(bm['engagement_index'])
    vec.append(bm['stress_score'])

    # [29-33] absolute alpha per channel
    for ch in ['AF3', 'T7', 'Pz', 'T8', 'AF4']:
        vec.append(bp['alpha'].get(ch, 0.0))

    # [34] frontal beta asymmetry
    eps = 1e-10
    if 'AF3' in ch_names and 'AF4' in ch_names:
        vec.append(float(np.log(bp['beta']['AF4'] + eps) - np.log(bp['beta']['AF3'] + eps)))
    else:
        vec.append(0.0)

    # Normalize to unit length (required for cosine similarity)
    v = np.array(vec, dtype=np.float32)
    norm = np.linalg.norm(v)
    if norm > 0:
        v = v / norm

    return v.tolist()

print('make_embedding defined — 35-dim biomarker vector')

make_embedding defined — 35-dim biomarker vector


In [7]:
# CELL 7: NARRATIVE GENERATOR

def generate_narrative(bp, bm, subject_id, condition_label, file_name):
    parts = []
    parts.append(
        f'EEG Clinical Narrative | Subject: {subject_id} | '
        f'Condition: {condition_label} | File: {file_name}'
    )

    faa = bm['frontal_alpha_asymmetry']
    if faa > 0.2:
        parts.append(f'Frontal Cortex: Right frontal alpha dominance (FAA={faa:.3f}). Stress, anxiety, withdrawal motivation.')
    elif faa < -0.2:
        parts.append(f'Frontal Cortex: Left frontal alpha dominance (FAA={faa:.3f}). Positive affect, approach motivation.')
    else:
        parts.append(f'Frontal Cortex: Symmetric frontal alpha (FAA={faa:.3f}). Neutral emotional state.')

    tar = bm['theta_alpha_ratio']
    if tar > 1.5:
        parts.append(f'Alertness: High Theta/Alpha ratio ({tar:.2f}) — cognitive fatigue, reduced alertness.')
    elif tar < 0.8:
        parts.append(f'Alertness: Low Theta/Alpha ratio ({tar:.2f}) — high alertness, relaxed wakefulness.')
    else:
        parts.append(f'Alertness: Moderate Theta/Alpha ratio ({tar:.2f}) — normal alertness.')

    ei = bm['engagement_index']
    if ei > 1.2:
        parts.append(f'Cognitive Load: High engagement index ({ei:.2f}) — active processing, possible stress.')
    elif ei < 0.6:
        parts.append(f'Cognitive Load: Low engagement index ({ei:.2f}) — relaxation or disengagement.')
    else:
        parts.append(f'Cognitive Load: Moderate engagement index ({ei:.2f}) — sustained cognitive activity.')

    band_summary = ' | '.join(
        f"{b.capitalize()}={np.mean(list(bp[b].values())):.4f}" for b in BANDS
    )
    parts.append(f'Spectral Profile: {band_summary}')

    ss = bm['stress_score']
    if ss > 0.6:
        stress_label = 'HIGH_STRESS'
        parts.append(f'Assessment: HIGH STRESS (score={ss}).')
    elif ss > 0.35:
        stress_label = 'MODERATE_STRESS'
        parts.append(f'Assessment: MODERATE STRESS (score={ss}).')
    else:
        stress_label = 'LOW_STRESS'
        parts.append(f'Assessment: LOW STRESS / RELAXED (score={ss}).')

    return '\n'.join(parts), stress_label

print('generate_narrative defined')

generate_narrative defined


In [8]:
# CELL 8: QUICK TEST — one file
test_file = next((DATASET_ROOT / CONDITIONS['CMPS']).glob('*.edf'))
print('Testing:', test_file.name)

data, ch_names = load_eeg(str(test_file))
bp  = compute_band_powers(data, ch_names)
bm  = compute_biomarkers(bp, ch_names)
emb = make_embedding(bp, bm, ch_names)
narrative, label = generate_narrative(bp, bm, '1', 'CMPS', test_file.name)

print('Embedding dims:', len(emb))
print('Stress label  :', label)
print()
print(narrative)

Testing: 1 (1).edf
Embedding dims: 30
Stress label  : MODERATE_STRESS

EEG Clinical Narrative | Subject: 1 | Condition: CMPS | File: 1 (1).edf
Frontal Cortex: Symmetric frontal alpha (FAA=-0.144). Neutral emotional state.
Alertness: High Theta/Alpha ratio (4.79) — cognitive fatigue, reduced alertness.
Cognitive Load: Low engagement index (0.15) — relaxation or disengagement.
Spectral Profile: Delta=0.0000 | Theta=0.0000 | Alpha=0.0000 | Beta=0.0000 | Gamma=0.0000
Assessment: MODERATE STRESS (score=0.4458).


In [ ]:
# CELL 9: REPLACE CHROMADB WITH SQLITE — zero dependencies

import sqlite3
import json
import numpy as np

DB_PATH = r"GIVE YOUR FILE LOCATION HERE\neurorag_db.sqlite"  # <-- change this to your desired path

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS eeg_narratives (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        subject_id  TEXT,
        condition   TEXT,
        file_name   TEXT,
        stress_label TEXT,
        stress_score REAL,
        tar         REAL,
        faa         REAL,
        ei          REAL,
        embedding   TEXT,
        narrative   TEXT
    )
''')
conn.commit()
print("SQLite DB ready:", DB_PATH)

SQLite DB ready: C:\Users\KARTHIKEYA\Desktop\neurorag.db


In [10]:
# CELL 10: MAIN LOOP — stores into SQLite instead of ChromaDB

import gc
import json

all_records = []
errors      = []
count       = 0

for condition_key, condition_folder in CONDITIONS.items():
    folder_path = DATASET_ROOT / condition_folder
    if not folder_path.exists():
        print(f'SKIPPING: {folder_path}')
        continue

    edf_files = sorted(folder_path.glob('*.edf'))
    print(f'\n[{condition_key}] {len(edf_files)} files')

    for edf_path in tqdm(edf_files, desc=condition_key):
        file_name  = edf_path.name
        subject_id = file_name.split(' ')[0]

        try:
            raw = mne.io.read_raw_edf(str(edf_path), preload=True, verbose=False)
            available = [ch for ch in EEG_CHANNELS if ch in raw.ch_names]
            raw_eeg = raw.copy().pick(available)
            raw_eeg.filter(l_freq=0.5, h_freq=45.0, verbose=False)
            raw_eeg.notch_filter(freqs=50.0, verbose=False)
            data = raw_eeg.get_data()
            del raw, raw_eeg
            gc.collect()

            bp  = compute_band_powers(data, available)
            bm  = compute_biomarkers(bp, available)
            emb = make_embedding(bp, bm, available)
            narrative, stress_label = generate_narrative(
                bp, bm, subject_id, condition_key, file_name
            )
            del data
            gc.collect()

            cursor.execute('''
                INSERT INTO eeg_narratives
                (subject_id, condition, file_name, stress_label,
                 stress_score, tar, faa, ei, embedding, narrative)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                subject_id,
                condition_key,
                file_name,
                stress_label,
                float(bm['stress_score']),
                round(float(bm['theta_alpha_ratio']), 4),
                round(float(bm['frontal_alpha_asymmetry']), 4),
                round(float(bm['engagement_index']), 4),
                json.dumps(emb),   # store embedding as JSON string
                narrative
            ))
            conn.commit()

            all_records.append({'subject_id': subject_id, 'condition': condition_key,
                                 'stress_label': stress_label})
            count += 1

        except Exception as e:
            errors.append({'file': str(edf_path), 'error': str(e)})
            gc.collect()

    gc.collect()

print(f'\nDONE. Stored {count} files in SQLite.')
print(f'Errors: {len(errors)}')
for e in errors:
    print(' ERR:', e['file'], '->', e['error'])


[CMPS] 22 files


CMPS: 100%|██████████| 22/22 [00:09<00:00,  2.39it/s]



[HORROR] 22 files


HORROR: 100%|██████████| 22/22 [00:08<00:00,  2.49it/s]



[MUSIC] 20 files


...


[SCWT] 24 files


SCWT: 100%|██████████| 24/24 [00:10<00:00,  2.20it/s]



[TMCT] 24 files


TMCT: 100%|██████████| 24/24 [00:15<00:00,  1.53it/s]


DONE. Stored 112 files in SQLite.
Errors: 0


In [11]:
# CELL 11: RAG RETRIEVAL from SQLite using cosine similarity

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

def retrieve(query_embedding, top_k=5):
    cursor.execute('SELECT subject_id, condition, stress_label, embedding, narrative FROM eeg_narratives')
    rows = cursor.fetchall()
    
    scored = []
    for subject_id, condition, stress_label, emb_json, narrative in rows:
        emb = json.loads(emb_json)
        sim = cosine_similarity(query_embedding, emb)
        scored.append((sim, subject_id, condition, stress_label, narrative))
    
    scored.sort(reverse=True)
    return scored[:top_k]

print("retrieve() ready")

retrieve() ready


In [ ]:
# CELL 12: OPENROUTER LLM + FULL RAG
import requests

OPENROUTER_API_KEY = "OPEN ROUTER API KEY"  # <-- paste your key here
MODEL = MODEL = "GPT-OSS-120B"

def llm(prompt):
    r = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
            "HTTP-Referer": "https://neurorag.research",
            "X-Title": "NeuroRAG"
        },
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 500,
            "temperature": 0.2
        },
        timeout=60
    )
    resp = r.json()
    # Show full response if something goes wrong
    if "choices" not in resp:
        print("API ERROR:", resp)
        return None
    return resp["choices"][0]["message"]["content"]

# Test API key first
test = llm("Reply with exactly: NeuroRAG online")
print("API test:", test)


API test: NeuroRAG online


In [24]:
# CELL 13: SQLITE STATS
cursor.execute("SELECT condition, stress_label, COUNT(*) FROM eeg_narratives GROUP BY condition, stress_label")
rows = cursor.fetchall()

total = cursor.execute("SELECT COUNT(*) FROM eeg_narratives").fetchone()[0]
print("Total docs in DB:", total)
print()
print("Condition    Stress Label         Count")
print("-" * 40)
for condition, stress_label, count in rows:
    print(condition, " " * (12 - len(condition)), stress_label, " " * (20 - len(stress_label)), count)

Total docs in DB: 336

Condition    Stress Label         Count
----------------------------------------
CMPS          HIGH_STRESS           9
CMPS          LOW_STRESS            21
CMPS          MODERATE_STRESS       36
HORROR        HIGH_STRESS           6
HORROR        LOW_STRESS            48
HORROR        MODERATE_STRESS       12
MUSIC         HIGH_STRESS           6
MUSIC         LOW_STRESS            36
MUSIC         MODERATE_STRESS       18
SCWT          HIGH_STRESS           24
SCWT          LOW_STRESS            24
SCWT          MODERATE_STRESS       24
TMCT          HIGH_STRESS           9
TMCT          LOW_STRESS            21
TMCT          MODERATE_STRESS       42


---
## RAG Pipeline with OpenRouter
Run cells 1-11 first, then run these.

In [ ]:
# CELL 14: INSTALL (run once)
# !pip install requests
print("ready")


ready


In [25]:
# CELL 14: FULL RAG PIPELINE — run on a real EEG file

def neurorag(edf_path, top_k=5):
    # Step 1: extract features from new EEG file
    data, ch_names   = load_eeg(edf_path)
    bp               = compute_band_powers(data, ch_names)
    bm               = compute_biomarkers(bp, ch_names)
    emb              = make_embedding(bp, bm, ch_names)
    narrative, label = generate_narrative(
        bp, bm, "QUERY", "UNKNOWN",
        edf_path.replace("\\", "/").split("/")[-1]
    )

    # Step 2: retrieve top_k similar cases from SQLite
    results = retrieve(emb, top_k)

    # Step 3: build context from retrieved cases
    context = ""
    for i, (sim, sid, cond, stress, doc) in enumerate(results):
        context += f"\n\n[Case {i+1} | Similarity:{sim:.3f} | Condition:{cond} | Stress:{stress}]\n{doc}"

    # Step 4: RAG prompt
    prompt = (
        "You are a clinical neuroscientist analyzing EEG brain signals.\n\n"
        "NEW EEG RECORDING TO ASSESS:\n" + narrative +
        "\n\nSIMILAR PAST EEG CASES RETRIEVED FROM DATABASE:" + context +
        "\n\nBased on the new EEG and the retrieved similar cases, provide:\n"
        "1. COGNITIVE STATE: What is the stress/cognitive state of this subject?\n"
        "2. EVIDENCE: Which biomarkers (TAR, FAA, Engagement Index) support this?\n"
        "3. CONFIDENCE: Low / Medium / High and why?\n"
        "4. RECOMMENDATION: One clinical action to take."
    )

    # Step 5: LLM reasons over query + retrieved context
    print("=" * 60)
    print("QUERY NARRATIVE:")
    print(narrative)
    print()
    print("TOP RETRIEVED CASES:")
    for sim, sid, cond, stress, _ in results:
        print(f"  Sim={sim:.3f} | Condition={cond} | Stress={stress} | Subject={sid}")
    print()
    print("LLM ASSESSMENT:")
    assessment = llm(prompt)
    print(assessment)
    print("=" * 60)
    return assessment

# Run on a real file
from pathlib import Path
query_file = str(DATASET_ROOT / CONDITIONS["CMPS"] / "1 (1).edf")
neurorag(query_file, top_k=5)


QUERY NARRATIVE:
EEG Clinical Narrative | Subject: QUERY | Condition: UNKNOWN | File: 1 (1).edf
Frontal Cortex: Symmetric frontal alpha (FAA=-0.144). Neutral emotional state.
Alertness: High Theta/Alpha ratio (4.79) — cognitive fatigue, reduced alertness.
Cognitive Load: Low engagement index (0.15) — relaxation or disengagement.
Spectral Profile: Delta=0.0000 | Theta=0.0000 | Alpha=0.0000 | Beta=0.0000 | Gamma=0.0000
Assessment: MODERATE STRESS (score=0.4458).

TOP RETRIEVED CASES:
  Sim=1.000 | Condition=CMPS | Stress=MODERATE_STRESS | Subject=1
  Sim=1.000 | Condition=CMPS | Stress=MODERATE_STRESS | Subject=1
  Sim=1.000 | Condition=CMPS | Stress=MODERATE_STRESS | Subject=1
  Sim=1.000 | Condition=TMCT | Stress=MODERATE_STRESS | Subject=3
  Sim=1.000 | Condition=TMCT | Stress=MODERATE_STRESS | Subject=3

LLM ASSESSMENT:
**1. COGNITIVE STATE**  
- **Moderate stress** with signs of **cognitive fatigue** and **reduced alertness**. The subject appears to be in a neutral‑emotional, relati

'**1. COGNITIVE STATE**  \n- **Moderate stress** with signs of **cognitive fatigue** and **reduced alertness**. The subject appears to be in a neutral‑emotional, relatively disengaged state rather than high‑arousal anxiety or deep relaxation.\n\n**2. EVIDENCE (key EEG biomarkers)**  \n\n| Biomarker | Value / Observation | Interpretation |\n|-----------|---------------------|----------------|\n| **Theta/Alpha Ratio (TAR)** | **4.79** (high) | Elevated TAR is a reliable marker of decreased vigilance and mental fatigue; it correlates with moderate‑to‑high stress levels. |\n| **Frontal Alpha Asymmetry (FAA)** | **‑0.144** (symmetric, slightly left‑dominant) | Near‑zero FAA indicates a neutral emotional tone; no strong left‑ or right‑hemispheric dominance that would suggest strong approach/withdrawal affect. |\n| **Engagement Index (EI)** | **0.15** (low) | Low EI reflects reduced cortical arousal and engagement, consistent with a relaxed or disengaged mental state and with moderate stress.

In [26]:
# CELL 15: BATCH EVALUATION — one file per condition
# Checks: does RAG retrieve correct condition? This is Table 1 in your paper.

import pandas as pd
from collections import Counter

print("NEURORAG BATCH EVALUATION")
print("=" * 60)

eval_rows = []

for condition_key, condition_folder in CONDITIONS.items():
    folder = DATASET_ROOT / condition_folder
    if not folder.exists():
        continue

    test_file = sorted(folder.glob("*.edf"))[0]
    print(f"\nCondition: {condition_key} | File: {test_file.name}")

    data, ch_names = load_eeg(str(test_file))
    bp  = compute_band_powers(data, ch_names)
    bm  = compute_biomarkers(bp, ch_names)
    emb = make_embedding(bp, bm, ch_names)

    results = retrieve(emb, top_k=5)
    retrieved_conds = [cond for _, _, cond, _, _ in results]
    top_voted = Counter(retrieved_conds).most_common(1)[0][0]
    correct   = top_voted == condition_key

    print(f"  Retrieved conditions : {retrieved_conds}")
    print(f"  Majority vote        : {top_voted}")
    print(f"  Correct              : {correct}")

    eval_rows.append({
        "true_condition"    : condition_key,
        "retrieved_majority": top_voted,
        "correct"           : correct
    })

df_eval = pd.DataFrame(eval_rows)
accuracy = df_eval["correct"].mean()
print(f"\nRetrieval Accuracy: {accuracy*100:.1f}%")
print()
print(df_eval.to_string(index=False))
df_eval.to_csv(r"C:\Users\KARTHIKEYA\Desktop\neurorag_eval.csv", index=False)
print("\nSaved: neurorag_eval.csv")


NEURORAG BATCH EVALUATION

Condition: CMPS | File: 1 (1).edf
  Retrieved conditions : ['CMPS', 'CMPS', 'CMPS', 'TMCT', 'TMCT']
  Majority vote        : CMPS
  Correct              : True

Condition: HORROR | File: 002 (1).edf
  Retrieved conditions : ['HORROR', 'HORROR', 'HORROR', 'HORROR', 'HORROR']
  Majority vote        : HORROR
  Correct              : True

Condition: MUSIC | File: 001 (1).edf
  Retrieved conditions : ['MUSIC', 'MUSIC', 'MUSIC', 'CMPS', 'CMPS']
  Majority vote        : MUSIC
  Correct              : True

Condition: SCWT | File: 2 (1).edf
  Retrieved conditions : ['SCWT', 'SCWT', 'SCWT', 'SCWT', 'SCWT']
  Majority vote        : SCWT
  Correct              : True

Condition: TMCT | File: 3 (1).edf
  Retrieved conditions : ['TMCT', 'TMCT', 'TMCT', 'MUSIC', 'MUSIC']
  Majority vote        : TMCT
  Correct              : True

Retrieval Accuracy: 100.0%

true_condition retrieved_majority  correct
          CMPS               CMPS     True
        HORROR             HO

In [27]:
# QUERY CELL — ask anything about your EEG data

import sqlite3, json
import numpy as np

# Reconnect if needed
conn   = sqlite3.connect(r"C:\Users\KARTHIKEYA\Desktop\neurorag.db")
cursor = conn.cursor()

# ── QUESTION 1: What is the overall stress level across all recordings? ──
cursor.execute("""
    SELECT stress_label, COUNT(*) as count,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM eeg_narratives), 1) as percent
    FROM eeg_narratives
    GROUP BY stress_label
    ORDER BY count DESC
""")
print("OVERALL STRESS DISTRIBUTION:")
print("-" * 40)
for label, count, pct in cursor.fetchall():
    bar = "█" * int(pct / 2)
    print(f"{label:<20} {count:>4} recordings  ({pct}%)  {bar}")

print()

# ── QUESTION 2: Which condition has highest stress? ──
cursor.execute("""
    SELECT condition,
           ROUND(AVG(stress_score), 3) as avg_stress,
           ROUND(MAX(stress_score), 3) as max_stress
    FROM eeg_narratives
    GROUP BY condition
    ORDER BY avg_stress DESC
""")
print("STRESS LEVEL PER CONDITION (highest first):")
print("-" * 50)
for cond, avg, mx in cursor.fetchall():
    bar = "█" * int(avg * 30)
    print(f"{cond:<10}  avg={avg}  max={mx}  {bar}")

print()

# ── QUESTION 3: Which subject is most stressed? ──
cursor.execute("""
    SELECT subject_id, condition,
           ROUND(AVG(stress_score), 3) as avg_stress
    FROM eeg_narratives
    GROUP BY subject_id, condition
    ORDER BY avg_stress DESC
    LIMIT 10
""")
print("TOP 10 MOST STRESSED (subject + condition):")
print("-" * 50)
for sid, cond, avg in cursor.fetchall():
    print(f"Subject {sid:<5} | {cond:<10} | stress={avg}")

OVERALL STRESS DISTRIBUTION:
----------------------------------------
LOW_STRESS            150 recordings  (44.6%)  ██████████████████████
MODERATE_STRESS       132 recordings  (39.3%)  ███████████████████
HIGH_STRESS            54 recordings  (16.1%)  ████████

STRESS LEVEL PER CONDITION (highest first):
--------------------------------------------------
SCWT        avg=0.502  max=0.767  ███████████████
CMPS        avg=0.428  max=0.779  ████████████
TMCT        avg=0.418  max=0.824  ████████████
MUSIC       avg=0.332  max=0.705  █████████
HORROR      avg=0.302  max=0.733  █████████

TOP 10 MOST STRESSED (subject + condition):
--------------------------------------------------
Subject 2     | SCWT       | stress=0.502
Subject 1     | CMPS       | stress=0.428
Subject 3     | TMCT       | stress=0.418
Subject 001   | MUSIC      | stress=0.332
Subject 002   | HORROR     | stress=0.302


In [28]:
# RAG QUERY CELL — type your question, get answer from EEG database

def ask_neurorag(question):
    """
    You type any question in plain English.
    NeuroRAG converts it to a query, retrieves similar EEG cases,
    and LLM answers based on retrieved evidence.
    """

    # Step 1: Retrieve relevant cases using keyword matching on narratives
    cursor.execute("SELECT subject_id, condition, stress_label, stress_score, tar, faa, ei, narrative FROM eeg_narratives")
    rows = cursor.fetchall()

    # Score each narrative by keyword relevance to your question
    question_lower = question.lower()
    keywords = question_lower.split()

    scored = []
    for sid, cond, stress, score, tar, faa, ei, narrative in rows:
        # Count how many question keywords appear in this narrative
        relevance = sum(1 for kw in keywords if kw in narrative.lower())
        # Also boost if question mentions condition directly
        if any(c.lower() in question_lower for c in [cond, stress]):
            relevance += 3
        scored.append((relevance, sid, cond, stress, score, tar, faa, ei, narrative))

    # Sort by relevance, take top 5
    scored.sort(reverse=True)
    top5 = scored[:5]

    # Step 2: Build context from retrieved cases
    context = ""
    for i, (rel, sid, cond, stress, score, tar, faa, ei, narrative) in enumerate(top5):
        context += (
            f"\n\n[Case {i+1} | Subject:{sid} | Condition:{cond} | "
            f"Stress:{stress} | Score:{score} | TAR:{tar} | FAA:{faa} | EI:{ei}]\n"
            f"{narrative}"
        )

    # Step 3: RAG prompt
    prompt = (
        f"You are a clinical neuroscientist with access to an EEG database.\n\n"
        f"USER QUESTION: {question}\n\n"
        f"RETRIEVED EEG CASES FROM DATABASE:\n{context}\n\n"
        f"Based on the retrieved EEG cases, answer the user's question clearly. "
        f"Reference specific biomarker values (TAR, FAA, EI, stress scores) in your answer. "
        f"Be specific, not generic."
    )

    # Step 4: LLM answers
    print("=" * 60)
    print(f"QUESTION: {question}")
    print("=" * 60)
    print("\nRELEVANT EEG CASES RETRIEVED:")
    for rel, sid, cond, stress, score, tar, faa, ei, _ in top5:
        print(f"  Subject:{sid} | {cond} | {stress} | stress_score={score}")
    print("\nNeuroRAG ANSWER:")
    answer = llm(prompt)
    print(answer)
    print("=" * 60)
    return answer

# ── ASK YOUR QUESTIONS HERE ──

ask_neurorag("What is the overall stress level in this dataset?")
ask_neurorag("Which condition causes the most stress?")
ask_neurorag("How stressed are subjects during horror video stimulation?")
ask_neurorag("Compare stress between math problem solving and relaxing music")
ask_neurorag("Which subjects show highest frontal alpha asymmetry?")
ask_neurorag("What does the theta alpha ratio tell us about cognitive fatigue here?")

QUESTION: What is the overall stress level in this dataset?

RELEVANT EEG CASES RETRIEVED:
  Subject:3 | TMCT | MODERATE_STRESS | stress_score=0.5109
  Subject:3 | TMCT | MODERATE_STRESS | stress_score=0.5109
  Subject:3 | TMCT | MODERATE_STRESS | stress_score=0.5109
  Subject:3 | TMCT | MODERATE_STRESS | stress_score=0.4949
  Subject:3 | TMCT | MODERATE_STRESS | stress_score=0.4949

NeuroRAG ANSWER:
**Overall stress level in the retrieved dataset**

All five EEG recordings are classified as **“MODERATE STRESS.”**  
The quantitative stress‑score values that drive this classification are clustered around the 0.5 – 0.51 range:

| Case | Stress score | TAR (Theta/Alpha) | FAA (Frontal‑alpha asymmetry) | EI (Engagement Index) |
|------|--------------|-------------------|------------------------------|-----------------------|
| 1‑3  | 0.5109       | 2.79 ± 0.00        | –0.221  (left‑dominant)      | 0.1715                |
| 4‑5  | 0.4949       | 2.27 ± 0.00        | +0.192  (symmetrical/r

'**Theta‑Alpha Ratio (TAR) as a marker of cognitive fatigue in the retrieved records**\n\n| Case | Subject | Condition | TAR | Stress score | FAA (frontal‑alpha asymmetry) | Engagement Index (EI) |\n|------|---------|-----------|-----|--------------|------------------------------|-----------------------|\n| 1‑3 | 3 | TMCT | **2.2235** | 0.8241 (high) | –0.998 (strong left‑frontal α dominance) | 0.6206 (moderate) |\n| 4‑5 | 1 | CMPS | **1.6612** | 0.779 (high) | –0.744 (left‑frontal α dominance) | 0.6489 (moderate) |\n\n### What the TAR tells us\n\n1. **Magnitude of TAR and fatigue**  \n   - In all five recordings the TAR is **well above 1.5**. In the neuro‑physiological literature, a TAR\u202f>\u202f1.5 is commonly interpreted as a shift of power from the faster **alpha (8‑12\u202fHz)** band toward the slower **theta (4‑7\u202fHz)** band. This shift reflects **reduced cortical arousal and the onset of cognitive fatigue**.  \n   - **Case\u202f1‑3 (TAR\u202f=\u202f2.22)** shows a *larger